# Phase 1 Evaluation — AI-Powered Visual Attendance System (Step 14)

## Evaluation objectives and scope

This notebook collects **reproducible, software-level evidence** for Phase 1:

1. environment and artifact availability (no downloads, ever),
2. how to re-run the automated regression suite safely,
3. deterministic contract checks of the implemented modules (detection,
   quality, recognition, temporal verification, unknown-face handling,
   attendance),
4. measured performance of the *software* paths that run on synthetic,
   non-biometric images, and
5. an explicit record of what could **not** be measured and why
   (missing model artifacts / labeled datasets / camera).

**Hard privacy rules honored by every cell:** no camera is opened at
notebook-run time, no employee/enrollment is created, the production
`data/` embedding store is untouched (all dirty experiments use private
temporary directories), and no biometric file is read, printed, or
copied. Nothing here downloads a dataset or a model artifact.

## What this notebook is *not*

Unless a cell explicitly runs a real model on real data (the only
opt-in is the local YuNet smoke check), its outputs are **contract and
correctness checks on synthetic inputs** — they are NOT empirical
computer-vision accuracy metrics. Detection precision/recall, quality
operating characteristics under real lighting, FAR/FRR recognition
curves, and spoof-resistance evaluation require consented, labeled,
real image datasets that are not present, and are recorded as
`NOT RUN`/`BLOCKED` in `reports/phase1_evaluation.md`.

## Environment & artifact availability

The project convention is *no side effects on import*; cells below follow it.

In [ ]:
import platform
import statistics
import sys, time
from pathlib import Path

import numpy as np
import cv2

from src.config import (
    PROJECT_ROOT, QUALITY, RECOGNITION, TEMPORAL_VERIFICATION, UNKNOWN_FACE,
    ATTENDANCE, AppSettings, QualityConfig, RecognitionConfig,
    TemporalVerificationConfig, UnknownFaceConfig, AttendanceConfig,
)
from src.face_detection import FaceDetection, FaceDetector
from src.quality import FaceQualityAssessor, FaceQualityDecision, FaceQualityReason
from src.embedding_storage import EmbeddingStorageService
from src.face_recognition import (
    FaceRecognitionService, RecognitionResult, RecognitionStatus,
)
from src.temporal_verification import (
    TemporalVerifier, TemporalVerificationResult, TemporalVerificationStatus,
)
from src.unknown_face_handler import (
    UnknownFaceHandler, UnknownFaceStatus, UnresolvedObservation,
)
from src.attendance import AttendanceEngine, AttendanceDecisionStatus, EventType

print("python:", sys.version.split()[0])
print("platform:", platform.platform())
print("numpy:", np.__version__, "| opencv:", cv2.__version__)
print("project root:", PROJECT_ROOT)
print()
print("component defaults (frozen, provisional until Phase 2 calibration):")
print("  QUALITY sharpness>=%.0f brightness=[%.0f,%.0f]" % (
    QUALITY.min_sharpness, QUALITY.min_brightness, QUALITY.max_brightness))
print("  RECOGNITION min_similarity=%.2f min_margin=%s" % (
    RECOGNITION.min_similarity, RECOGNITION.min_margin))
print("  TEMPORAL min_consistent_frames=%d window_frames=%d" % (
    TEMPORAL_VERIFICATION.min_consistent_frames,
    TEMPORAL_VERIFICATION.window_frames))
print("  UNKNOWN_FACE cooldown=%.0fs max_sessions=%d" % (
    UNKNOWN_FACE.episode_cooldown_seconds, UNKNOWN_FACE.max_sessions))
print("  ATTENDANCE check_in_cooldown=%.0fs max_tracked=%d" % (
    ATTENDANCE.check_in_cooldown_seconds, ATTENDANCE.max_tracked_employees))


In [ ]:
# Artifact availability: operator-supplied local files only. This notebook
# never downloads anything; a missing artifact is reported, not fetched.
models_dir = PROJECT_ROOT / "models"
artifacts = {
    "yunet.onnx (detection model)": (models_dir / "yunet.onnx").is_file(),
    "sface.onnx (embedding model)": (models_dir / "sface.onnx").is_file(),
}
for name, present in artifacts.items():
    print(("PRESENT  " if present else "MISSING  ") + name)
print()
print("Recognition (Steps 08-10) end-to-end needs sface.onnx; without it all")
print("embedding-dependent empirical evaluation is BLOCKED. Detection")
print("(Steps 05/06) can run against the local artifact (opt-in cell below).")


## Test configuration & reproducibility

The suite is deterministic (no sleeps, no camera, no network). From the
**project root**:

```
python -m pytest -q        # currently 654 passed
```

From **outside** the project root (established project requirement):

```
cd /tmp/any-dir
<project>/.venv/Scripts/python.exe -m pytest <project>/tests -q
```

`pytest.ini` sets `testpaths = tests` and `pythonpath = .`.

In [ ]:
# Opt-in regression runner (default OFF: the notebook run stays fast and
# independent of the test suite). Set to True to run the suite here.
RUN_FULL_TEST_SUITE = False
if RUN_FULL_TEST_SUITE:
    import subprocess
    r = subprocess.run(
        [sys.executable, "-m", "pytest", "-q"],
        capture_output=True, text=True, timeout=600,
        cwd=str(PROJECT_ROOT),
    )
    print(r.stdout[-3000:])
else:
    print("RUN_FULL_TEST_SUITE=False -> suite not run here; use the commands above.")


## Face-detection evaluation (software level)

Checks below use only synthetic frames and pure dataclass validation —
no claims about accuracy on real images. Ground-truth-annotated
datasets are unavailable, so detection **precision / recall is NOT
RUN** (see `reports/phase1_evaluation.md`).

In [ ]:
# Box-model contract: positive sizes, non-negative origin, [0,1] confidence.
ok = FaceDetection(x=1, y=2, width=3, height=4, confidence=0.9)
for bad in (dict(x=1, y=2, width=0, height=4),
            dict(x=-1, y=0, width=3, height=4),
            dict(x=0, y=0, width=3, height=4, confidence=1.5)):
    try:
        FaceDetection(**bad)
        print("UNEXPECTED: accepted", bad)
    except Exception as exc:
        print("rejected:", type(exc).__name__)


def iou(a: FaceDetection, b: FaceDetection) -> float:
    """Documented matching criterion for future labeled evaluation: IoU."""
    x1, y1 = max(a.x, b.x), max(a.y, b.y)
    x2 = min(a.x + a.width, b.x + b.width)
    y2 = min(a.y + a.height, b.y + b.height)
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = a.width * a.height + b.width * b.height - inter
    return inter / union if union else 0.0


a = FaceDetection(x=10, y=10, width=50, height=50)
b = FaceDetection(x=20, y=20, width=50, height=50)
print("IoU overlap example:", round(iou(a, b), 4),
      "| disjoint:", iou(a, FaceDetection(x=200, y=200, width=50, height=50)))


In [ ]:
# Opt-in YuNet smoke check: loads the LOCAL operator-supplied yunet.onnx
# (no download) and runs inference on a SYNTHETIC image (no camera, no
# biometric data, nothing persisted). Verifies artifact sanity and gives a
# synthetic-image latency figure ONLY (not an accuracy claim).
RUN_YUNET_SMOKE_CHECK = True  # <-- explicit opt-in for the local artifact
if RUN_YUNET_SMOKE_CHECK and artifacts["yunet.onnx (detection model)"]:
    detector = FaceDetector()
    detector.load()
    print("model loaded:", detector.is_loaded)
    synthetic = np.zeros((240, 320, 3), dtype=np.uint8)  # uniform: no face
    dets = detector.detect(synthetic)
    print("detections on uniform synthetic frame:", len(dets), "(expected 0)")
    for _ in range(10):
        detector.detect(synthetic)  # warm-up
    times = []
    for _ in range(100):
        t0 = time.perf_counter()
        detector.detect(synthetic)
        times.append(time.perf_counter() - t0)
    print("median detection latency on 240x320 synthetic frame: %.2f ms" % (
        statistics.median(times) * 1000.0))
else:
    print("BLOCKED: yunet.onnx absent (or opt-in disabled) -> no real detector run.")


## Face-quality evaluation (software level, synthetic inputs)

No real camera/lighting data exists here — the frames below are
deterministic synthetic images, so the decisions shown exercise
the *policy thresholds*; they do not characterize behavior on real
faces. The pose check is a documented **proxy** (landmark-offset
ratio), NOT a 3-D yaw/pitch angle.

In [ ]:
assessor = FaceQualityAssessor(QUALITY)
BOX = FaceDetection(x=0, y=0, width=112, height=112)


def make_frame(kind: str) -> np.ndarray:
    """Deterministic synthetic 240x320 BGR (H, W, 3) frame; saved NOWHERE."""
    img = np.zeros((240, 320), dtype=np.uint8)
    if kind == "dark":
        img = img + 10                                  # mean ~10 (below 40)
    elif kind == "bright":
        img = img + 245                                 # mean ~245 (above 220)
    elif kind == "flat":
        img = img + 128                                 # zero Laplacian variance
    else:  # "textured": deterministic checkerboard, mean ~128, huge variance
        yy, xx = np.mgrid[0:240, 0:320]
        img[(yy // 4 + xx // 4) % 2 == 0] = 150
        img[(yy // 4 + xx // 4) % 2 == 1] = 106
    return np.repeat(img[:, :, np.newaxis], 3, axis=2)


for kind in ("dark", "bright", "flat", "textured"):
    q = assessor.assess_face(make_frame(kind), BOX)
    reason_name = q.reason.value if q.reason is not None else "None"
    print(
        f"{kind:>8}: {q.decision.value:>6} reason={reason_name:<20} "
        f"sharpness={q.sharpness:8.2f} brightness={q.brightness:6.2f}"
    )


# Frame face-count policy (independent of any model):
d1 = FaceDetection(x=0, y=0, width=80, height=80)
d2 = FaceDetection(x=120, y=0, width=80, height=80)
print("policy one face  :", assessor.assess_frame_faces([d1]))
print("policy zero faces:", assessor.assess_frame_faces([]).value)
print("policy two faces :", assessor.assess_frame_faces([d1, d2]).value)


## Face-recognition evaluation (software level, synthetic identities)

Threshold/decision behavior is demonstrated on **synthetic orthogonal
unit vectors** with a scriptable embedder and a private temporary
storage directory. These rows show the *decision procedure* working —
they are **not** identity accuracy, FAR/FRR, or calibrated
probabilities (`NOT MEASURED`: no sface.onnx, no labeled dataset). No
identity leakage exists because there is no such split to leak.

In [ ]:
from types import SimpleNamespace
import tempfile, shutil, json
from src.enrollment import MANIFEST_FILENAME


class FakeEmbedder:
    """Scriptable Step 08 double: returns a fixed validated vector."""

    def __init__(self, vector):
        self.vector = vector

    def embed(self, crop):
        return SimpleNamespace(vector=self.vector.copy())


def unit(dim, index):
    v = np.zeros(dim, dtype=np.float32)
    v[index] = 1.0
    return v


tmp = Path(tempfile.mkdtemp(prefix="eval-recognition-"))
try:
    settings = AppSettings(data_dir=tmp / "data", log_dir=tmp / "logs")
    storage = EmbeddingStorageService(settings=settings)

    def enroll(employee_id):
        """Mirror the Step 07 eligibility marker (as the tests do)."""
        d = settings.employees_dir / employee_id
        d.mkdir(parents=True, exist_ok=True)
        manifest = {
            "format_version": 1,
            "employee": {
                "employee_id": employee_id,
                "is_active": True,
            },
            "sample_count": 1,
            "samples": [],
        }
        (d / MANIFEST_FILENAME).write_text(
            json.dumps(manifest), encoding="utf-8"
        )

    storage.save("A", unit(8, 0));   enroll("A")
    storage.save("B", unit(8, 1));   enroll("B")
    crop = np.zeros((16, 16, 3), dtype=np.uint8)

    # Exact-match query against enrolled A (cosine 1.0):
    svc = FaceRecognitionService(
        embedder=FakeEmbedder(unit(8, 0)), storage=storage,
        config=RecognitionConfig(min_similarity=0.36), settings=settings)
    r = svc.recognize(crop)
    print("query==A    ->", r.status.value, "id=", r.employee_id,
          "best=", round(r.best_score, 3) if r.best_score is not None else None)

    # Orthogonal query (cosine 0.0 < 0.36):
    svc_others = FaceRecognitionService(
        embedder=FakeEmbedder(unit(8, 2)), storage=storage,
        config=RecognitionConfig(min_similarity=0.36), settings=settings)
    r = svc_others.recognize(crop)
    print("query=C     ->", r.status.value, "id=", r.employee_id,
          "best=", round(r.best_score, 3) if r.best_score is not None else None)

    # Ambiguous top-two (identical scores, margin 0 < min_margin=0.1):
    storage.save("A2", 2.0 * unit(8, 0) + unit(8, 1)); enroll("A2")
    storage.save("B2", 2.0 * unit(8, 0) + unit(8, 2)); enroll("B2")
    svc_amb = FaceRecognitionService(
        embedder=FakeEmbedder(unit(8, 0)), storage=storage,
        config=RecognitionConfig(min_similarity=0.36, min_margin=0.1),
        settings=settings)
    r = svc_amb.recognize(crop, candidate_employee_ids=["A2", "B2"])
    print("tied top-2  ->", r.status.value, "id=", r.employee_id)
finally:
    shutil.rmtree(tmp, ignore_errors=True)   # temporary store removed


## Temporal verification, unknown-face handling, attendance correctness

Deterministic behavior checks with injected fake clocks (no sleeps).

In [ ]:
# 1) Temporal verifier (Step 11 API: consumes raw RecognitionResult frames
#    one at a time; the verifier itself returns COLLECTING/VERIFIED/INTERRUPTED).
verifier = TemporalVerifier(
    config=TemporalVerificationConfig(min_consistent_frames=2)
)
timeline = []
for frame in ("matched", "matched", "unknown_interrupt", "matched", "matched"):
    if frame == "matched":
        result = verifier.update(RecognitionResult(
            status=RecognitionStatus.MATCHED, employee_id="007"))
    else:
        result = verifier.update(RecognitionResult(
            status=RecognitionStatus.UNKNOWN))
    timeline.append("%s[%s]" % (result.status.value,
                                result.employee_id or "-"))
print("temporal timeline:", " -> ".join(timeline))


In [ ]:
# 2) Unknown-face handler: alert -> suppression -> expiry -> re-alert,
#    with an injected fake clock (no sleeping).
class FakeClock:
    def __init__(self):
        self.now = 0.0

    def __call__(self):
        return self.now


clock = FakeClock()
handler = UnknownFaceHandler(
    config=UnknownFaceConfig(episode_cooldown_seconds=10.0), clock=clock)
o1 = handler.observe_recognition(
    RecognitionResult(status=RecognitionStatus.UNKNOWN))
o2 = handler.observe_recognition(
    RecognitionResult(status=RecognitionStatus.AMBIGUOUS))
clock.now = 10.0   # exactly alert + cooldown -> episode expired deterministically
o3 = handler.observe_recognition(
    RecognitionResult(status=RecognitionStatus.NO_CANDIDATES))
print("unknown-face timeline:",
      (o1.status.value, o2.status.value), "then",
      (o3.status.value, o3.observation.value))


In [ ]:
# 3) Attendance engine: suppression boundary, identity preservation,
#    and inactive-employee gating (injected clocks, no sleeps).
from datetime import UTC, datetime


class FakeClocks:
    def __init__(self):
        self.mono, self.wall = 0.0, datetime(2026, 10, 13, 9, 0, tzinfo=UTC)


clocks = FakeClocks()
engine = AttendanceEngine(
    config=AttendanceConfig(check_in_cooldown_seconds=5.0),
    monotonic_clock=lambda: clocks.mono,
    clock_wall=lambda: clocks.wall,
)


def verified(emp="007"):
    return TemporalVerificationResult(
        status=TemporalVerificationStatus.VERIFIED, employee_id=emp)


d1 = engine.update(verified("007"))
clocks.mono = 4.0
d2 = engine.update(verified("007"))
clocks.mono = 5.0
d3 = engine.update(verified("007"))
d4 = engine.update(verified("EMP-0042"))
inactive = AttendanceEngine(
    config=AttendanceConfig(check_in_cooldown_seconds=5.0),
    monotonic_clock=lambda: 0.0, clock_wall=lambda: clocks.wall,
    active_lookup=lambda emp: False).update(verified("X"))

print("first             :", d1.status.value, d1.event.employee_id,
      "at", d1.event.occurred_at.isoformat())
print("within cooldown   :", d2.status.value, "event:", d2.event)
print("at exact boundary :", d3.status.value)
print("other employee    :", d4.status.value, d4.event.employee_id)
print("inactive gated    :", inactive.status.value, "event:", inactive.event)


## Performance measurements (software paths)

`time.perf_counter` after warm-up, median over repeats, synthetic
inputs. These measure **software overhead**, not model inference:
end-to-end camera-to-event latency and SFace embedding time are
`NOT MEASURED` (no artifact). Absolute numbers are machine-dependent;
only orders of magnitude are meaningful here.

In [ ]:
def median_us(fn, repeats=500, warmup=20):
    for _ in range(warmup):
        fn()
    samples = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        samples.append((time.perf_counter() - t0) * 1e6)
    return statistics.median(samples)


from src.face_recognition import cosine_similarity

va, vb = unit(4, 0), unit(4, 1)
tex_frame = make_frame("textured")
check = FaceDetection(x=0, y=0, width=112, height=112)

print("cosine_similarity       : %8.1f us" % median_us(lambda: cosine_similarity(va, vb)))
print("quality assess_face     : %8.1f us" % median_us(
    lambda: assessor.assess_face(tex_frame, check)))
print("temporal update (fresh) : %8.1f us" % median_us(
    lambda: TemporalVerifier(config=TemporalVerificationConfig()).update(
        RecognitionResult(
            status=RecognitionStatus.MATCHED, employee_id="007"))))


## Limitations, spoof-resistance boundary, and conclusions

**Untested/blocked (do NOT infer from synthetic results):**

| Area | Status | Reason |
|---|---|---|
| Detection precision / recall | NOT RUN | no ground-truth annotated dataset |
| Detection accuracy on real faces | NOT RUN | synthetic frames contain no faces |
| Quality behavior under real lighting/pose | NOT RUN | no consented labeled images |
| Recognition FAR / FRR / unknown-rate | NOT MEASURED | sface.onnx missing; no labeled dataset |
| End-to-end camera latency | NOT RUN | no camera at notebook run time (Step 15 scope) |
| Spoof resistance (print/replay/video) | NOT RUN | no liveness component (later phase); no authorized spoof rig |

**Documented spoof scenario plan (requires consented data and explicit
authorization; none executed here):** printed photos, phone/tablet
screens, and video replay at varying angles, recording acceptance /
rejection per scenario. **No spoof-resistance claim is made** — no
liveness detection exists in Phase 1 by design.

**Conclusions drawn from the *measured* evidence:**

1. The full software chain (recognition decision → temporal
   verification → unknown-face handling → attendance event) behaves
   deterministically on synthetic inputs, including exact
   suppression/cooldown boundaries and fail-safe rejection of
   malformed inputs.
2. Duplicate check-in suppression and its exact boundary are verified
   in-software; durable or cross-process exactly-once remains out of
   scope.
3. Empirical CV evaluation is blocked on operator-supplied artifacts
   (`sface.onnx`) and consented labeled data; every such metric is
   recorded as blocked in `reports/phase1_evaluation.md`.